# Bounded-alpha abundant timing baseline

Fresh bounded-alpha abundant-data timing baseline with worker shards prevented from racing on the final global completion marker.


In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

HERE = Path.cwd()
BASE_NOTEBOOK = HERE / "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed.ipynb"
if not BASE_NOTEBOOK.exists():
    raise FileNotFoundError(f"Missing committed abundant-data base notebook: {BASE_NOTEBOOK}")

print("Bounded-alpha abundant timing baseline base:", BASE_NOTEBOOK.name)

base_nb = json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))
patched_study = False
patched_results = False
verified_bounded_identifier = False
patched_cells = []

for cell_index, cell in enumerate(base_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if "identify_nonlinear_unbounded" in src:
        raise RuntimeError("Base abundant notebook unexpectedly references unbounded identifier.")
    if "opinion_dynamics.identify_nonlinear" in src:
        verified_bounded_identifier = True

    src2, n = re.subn(
        r'STUDY_NAME\s*=\s*"[^"]+"',
        'STUDY_NAME = "baseline_b_ab"',
        src,
        count=1,
    )
    if n:
        src = src2
        patched_study = True

    src, _ = re.subn(
        r'PIPELINE_VERSION\s*=\s*"[^"]+"',
        'PIPELINE_VERSION = "2026-09-08-bounded-abundant-timing-v2"',
        src,
        count=1,
    )

    old_name = "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed"
    if old_name in src:
        src = src.replace(old_name, "baseline_b_ab")
        patched_results = True

    patched_cells.append((cell_index, src))

required = {
    "bounded identifier import": verified_bounded_identifier,
    "fresh short STUDY_NAME": patched_study,
    "fresh short results namespace": patched_results,
}
missing = [k for k,v in required.items() if not v]
if missing:
    raise RuntimeError("Expected abundant benchmark structure missing: " + ", ".join(missing))

print("Bounded abundant overrides validated.")
worker_mode = os.environ.get("ABUNDANT_FULL_SHARD_ID") not in (None, "")
print("Driver mode:", "worker" if worker_mode else "final validation/merge")

cells_to_run = patched_cells[:-1] if worker_mode else patched_cells
if worker_mode:
    print("Worker mode: skipping base notebook final global validation/merge cell.")

g = globals()
for cell_index, src in cells_to_run:
    print(f"[base code cell {cell_index}]")
    exec(compile(src, f"{BASE_NOTEBOOK.name}:cell_{cell_index}", "exec"), g, g)
